In [ ]:
import pandas as pd
import numpy as np
import math as mt
import os
import pandas as pd
import matplotlib.pyplot as plt
import sys
import csv
import importlib

from sklearn.metrics import mean_squared_error

sys.path.append(os.path.abspath('.'))  # REDLAB_UQ.py is in the project root

# Load (and reload) the REDLAB_UQ module
try:
    from REDLAB_UQ import SurrogateModel
    # Reload the module if it is already loaded
    import REDLAB_UQ
    importlib.reload(REDLAB_UQ)
    from REDLAB_UQ import SurrogateModel
    print("REDLAB_UQ module loaded.")
except ImportError as e:
    print(f"Failed to load the REDLAB_UQ module: {e}")
    print("Check that REDLAB_UQ.py is in the project root folder.")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
## Input sampling
import numpy as np
from scipy.stats import qmc, norm

def sample_lognormal_rqmc(means, stds, corr, nsampleO, seed=None):
    """
    means:   (N,)   physical means μ_X (e.g., [30, 50, 10, ...])
    stds:    (N,)   physical standard deviations σ_X
    corr:    (N,N) correlation matrix in the normal (Z) space (symmetric, diag=1)
    nsampleO: number of samples
    """
    means = np.array(means, dtype=float)
    stds  = np.array(stds, dtype=float)
    N = len(means)

    # 1) (μ_Z, σ_Z) of Z from (μ_X, σ_X) of X
    #    σ_Z^2 = ln(1 + (σ_X/μ_X)^2)
    #    μ_Z   = ln(μ_X) - 0.5*σ_Z^2
    var_ratio = (stds / means)**2
    sigma_Z2  = np.log(1.0 + var_ratio)
    sigma_Z   = np.sqrt(sigma_Z2)
    mu_Z      = np.log(means) - 0.5 * sigma_Z2

    # 2) Covariance matrix in the Z space, Σ_Z = D * corr * D (D = diag(σ_Z))
    D = np.diag(sigma_Z)
    cov_Z = D @ corr @ D

    # 3) RQMC uniform samples on [0,1]^N (scrambled Sobol')
    engine = qmc.Sobol(d=N, scramble=True, seed=seed)
    engine.fast_forward(1024)     # skip the initial points (optional)
    U = engine.random(nsampleO)  # (nsampleO, N)

    # 4) U -> standard normal N(0,1) (independent samples in the normal space)
    eps = np.finfo(float).eps
    Z_indep = norm.ppf(np.clip(U, eps, 1 - eps))

    # 5) Introduce the correlation by Cholesky factorization: Z_corr ~ N(0, Σ_Z)
    L = np.linalg.cholesky(cov_Z)
    Z_corr = Z_indep @ L.T    # (nsampleO, N)

    # 6) Z_corr + μ_Z -> lognormal X = exp(Z_corr + μ_Z)
    X = np.exp(Z_corr + mu_Z)

    return X

base_values = [1] * 6  # normalized (z-scale) samples, so the means are 1
std_devs = [0.02, 0.02, 0.02, 0.02, 0.05, 0.025]  # COVs: 1-4 geometry (0.02), 5 Young's modulus (0.05), 6 tensile strength (0.025)

rho1 = 0.4  # correlation among the geometric variables (variables 1-4)
rho2 = 0.05    # correlation between Young's modulus and tensile strength (variables 5-6)

# 6x6 correlation matrix
NN = 6
corr = np.eye(NN)  # initialize with ones on the diagonal

# Correlation among the 1st-4th (geometric) variables: rho1
for i in range(4):
    for j in range(4):
        if i != j:
            corr[i, j] = rho1

# Correlation between the 5th-6th variables (Young's modulus - tensile strength): rho2
corr[4, 5] = rho2
corr[5, 4] = rho2

# Print the correlation matrix
print("Correlation matrix:")
print("="*60)
print("Variables: 1-4 (geometry), 5 (Young's modulus), 6 (tensile strength)")
print("-"*60)
print(corr)
print("="*60)
print(f"\nCorrelation among the geometric variables (1-4): {rho1}")
print(f"Correlation between Young's modulus and tensile strength (5-6): {rho2}\n")

# Covariance matrix and comparison
print("Covariance matrix:")
print("="*60)
D = np.diag(std_devs)  # diagonal matrix of standard deviations
cov_matrix = D @ corr @ D  # Cov = D × Corr × D
print("Covariance matrix:")
print(cov_matrix)
print("="*60)

# Comparison
print("\nCorrelation matrix vs covariance matrix:")
print(f"corr[0,1] (correlation) = {corr[0,1]:.6f}")
print(f"cov[0,1] (covariance)  = {cov_matrix[0,1]:.6f}")
print(f"-> The values differ because the standard deviations differ, although the means are 1.")
print(f"\ncorr[0,0] (diagonal, always 1) = {corr[0,0]:.6f}")
print(f"cov[0,0] (diagonal, variance σ²) = {cov_matrix[0,0]:.6f} = {std_devs[0]}² = {std_devs[0]**2:.6f}")
print("="*60)





In [ ]:
X_samples = sample_lognormal_rqmc(base_values, std_devs, corr, nsampleO=1_000_000, seed=123457)

# Save X_samples as CSV
import pandas as pd

# Convert the numpy array to a DataFrame
df_samples = pd.DataFrame(X_samples, columns=[f'X_{i+1}' for i in range(6)])

# Save as CSV
output_path = 'initial/input_samples_zz.csv'
df_samples.to_csv(output_path, index=False)

print(f"Samples saved to '{output_path}'.")
print(f"Shape: {df_samples.shape}")
print(f"\nFirst 5 rows:")
print(df_samples.head())

# Verification of the sampling
- Check the means and standard deviations of the 1,000,000 samples
- Target: unit means and the prescribed standard deviations (normalized samples)

In [ ]:
import pandas as pd

# Mean and standard deviation of each variable (full precision)
summary = []
for i in range(6):
    mean_sample = X_samples[:, i].mean()
    std_sample = X_samples[:, i].std()
    summary.append({
        'Variable': f'X_{i+1}',
        'Target Mean': base_values[i],
        'Sample Mean': f'{mean_sample:.10f}',  # 10 decimal places
        'Target Std': std_devs[i],
        'Sample Std': f'{std_sample:.10f}',  # 10 decimal places
        'Mean Error (%)': abs(mean_sample - base_values[i]) / base_values[i] * 100,
        'Std Error (%)': abs(std_sample - std_devs[i]) / std_devs[i] * 100
    })

df_summary = pd.DataFrame(summary)
print("=== Statistics of all samples (1,000,000) ===")
print(df_summary.to_string(index=False))
print(f"\nNumber of samples: {X_samples.shape[0]:,}")
print(f"Number of variables: {X_samples.shape[1]}")

In [ ]:
# Statistics of the first L samples
L_values = [26] #[100, 500, 1000, 5000, 10000, 50000, 100000]

print("\n" + "="*100)
print("=== Means and standard deviations of the first L samples ===")
print("="*100)

# Collect all results in one DataFrame
all_results = []

for L in L_values:
    for i in range(6):
        mean_L = X_samples[:L, i].mean()
        std_L = X_samples[:L, i].std()
        all_results.append({
            'L': L,
            'Variable': f'X_{i+1}',
            'Mean': f'{mean_L:.6f}',
            'Std': f'{std_L:.6f}',
            'Mean Error(%)': f'{abs(mean_L - base_values[i]) / base_values[i] * 100:.4f}',
            'Std Error(%)': f'{abs(std_L - std_devs[i]) / std_devs[i] * 100:.4f}'
        })

df_all = pd.DataFrame(all_results)

# Print grouped by L
for L in L_values:
    print(f"\n{'='*80}")
    print(f"First {L:,} samples")
    print(f"{'='*80}")
    df_L = df_all[df_all['L'] == L].drop('L', axis=1)
    print(df_L.to_string(index=False))

In [ ]:
mean_values = [36, 36, 70, 5, 69, 305]
#mean_values = [12, 12, 30, 9, 70, 305]

# Load the normalized input samples
df_zz = pd.read_csv('initial/input_samples_zz.csv')

# Multiply each column by mean_values
df_multiplied = df_zz * mean_values

# Select the first L samples
df_top_L = df_multiplied.head(L)

# Print the minimum and maximum of each variable
print(f"\n{'='*80}")
print(f"Minimum/maximum of each variable over the first {L} samples")
print(f"{'='*80}")

for i, col in enumerate(df_top_L.columns):
    min_val = df_top_L[col].min()
    max_val = df_top_L[col].max()
    print(f"{col}: Min = {min_val:.6f}, Max = {max_val:.6f}")

print(f"{'='*80}")

In [ ]:
import matplotlib.pyplot as plt
import math

# Histograms
ncols = 2
nrows = math.ceil(10 / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3*nrows))
axes = axes.flatten()

# Convert df_multiplied to a numpy array
X_samples_multiplied = df_multiplied.to_numpy()

# mean_values of each variable
plot_means = mean_values[:6]  # mean_values of X_1 to X_6

for i in range(6):
    axes[i].hist(X_samples_multiplied[:, i], bins=100, color='skyblue', edgecolor='k', alpha=0.7, density=True)
    
    # Target mean line
    axes[i].axvline(mean_values[i], color='red', linestyle='--', linewidth=2, label=f'Target Mean={mean_values[i]}')
    
    # Sample mean line
    actual_mean = X_samples_multiplied[:, i].mean()
    axes[i].axvline(actual_mean, color='green', linestyle=':', linewidth=2, label=f'Sample Mean={actual_mean:.2f}')
    
    axes[i].set_title(f'X_{i+1}')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    #axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()